# Segment-dependent systematics

This tutorial injects a campaign-like variance jump and shows both the visible failure mode and the exact-window calibration used by the segment veto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from urdr import (
    SegmentSystematicConfig,
    SimulationConfig,
    add_segment_systematics,
    benchmark_segment_veto,
    make_observing_window,
    segment_diagnostics,
    simulate_time_series,
)

In [ ]:
window = make_observing_window(
    duration_days=1.0,
    cadence_seconds=120.0,
    gaps_days=((0.49, 0.51),),
)
simulation = SimulationConfig(
    white_noise_sigma=0.2,
    granulation_amplitude=0.1,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=350.0,
    oscillation_amplitude=1.5,
)
segments = ((0.0, 0.5), (0.5, 1.0))
variance_jump = [SegmentSystematicConfig(0.5, 1.0, amplitude_scale=4.0)]
noise = simulate_time_series(
    window, simulation, np.random.default_rng(4), include_oscillations=False
)
contaminated = add_segment_systematics(noise, variance_jump)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, sharey=True)
for ax, series, title in [
    (axes[0], noise, "Stationary null"),
    (axes[1], contaminated, "Injected variance jump"),
]:
    ax.plot(series.time[series.observed], series.flux[series.observed], lw=0.65)
    ax.axvline(0.5, color="tab:red", ls="--", label="segment boundary")
    ax.axvspan(0.49, 0.51, color="0.85", label="gap")
    ax.set(ylabel="Flux", title=title)
axes[1].set_xlabel("Time [days]")
axes[0].legend(ncols=2)
plt.tight_layout()
plt.show()

{
    "stationary": segment_diagnostics(noise, segments),
    "variance_jump": segment_diagnostics(contaminated, segments),
}

The diagnostic combines robust scale ratios, median offsets, and within-segment drift. These quantities have different natural scales, so their joint threshold is learned from simulations rather than read directly from this plot.

In [ ]:
result = benchmark_segment_veto(
    window=window,
    simulation=simulation,
    systematics={"variance_jump": variance_jump},
    segments_days=segments,
    centre_frequencies_uhz=np.array([900.0, 1000.0, 1100.0]),
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=np.array([90.0, 100.0, 110.0]),
    realizations=8,
    target_false_positive_rate=0.25,
    target_signal_retention=0.9,
    max_lag_seconds=15_000.0,
    seed=42,
)
{
    "signal_before": result.raw_signal_detection_rate,
    "signal_after": result.vetoed_signal_detection_rate,
    "systematic_before": result.raw_systematic_detection_rate,
    "systematic_after": result.vetoed_systematic_detection_rate,
}

The raw and vetoed rates should always be reported together. A useful veto removes systematic detections while preserving the requested fraction of genuine EACF detections; its threshold is target- and window-specific.